# Part C - Pandas

In [1]:
import numpy as np
import pandas as pd

## C1 - Series and DataFrame by hand

In [2]:
names = ["Aarav", "Bibek", "Chandni", "Deepika", "Esha"]
marks_list = [72, 88, 55, 91, 63]

marks_series = pd.Series(marks_list, index=names)
print("C1 - Series index:", marks_series.index.tolist())
print("C1 - Series values:", marks_series.values)

# same small table built two different ways
df_from_dict = pd.DataFrame({"name": names, "marks": marks_list})
df_from_records = pd.DataFrame([{"name": n, "marks": m} for n, m in zip(names, marks_list)])

print("C1 - both ways give the same table:", df_from_dict.equals(df_from_records))
print("C1 - shape:", df_from_dict.shape)
print("C1 - columns:", df_from_dict.columns.tolist())
print("C1 - index:", df_from_dict.index.tolist())

C1 - Series index: ['Aarav', 'Bibek', 'Chandni', 'Deepika', 'Esha']
C1 - Series values: [72 88 55 91 63]
C1 - both ways give the same table: True
C1 - shape: (5, 2)
C1 - columns: ['name', 'marks']
C1 - index: [0, 1, 2, 3, 4]


## C2 - Load and inspect

In [3]:
df = pd.read_csv("../data/scores_raw.csv")

print("C2 - head:\n", df.head())
print("C2 - tail(3):\n", df.tail(3))
print("C2 - shape:", df.shape)
df.info()
print("C2 - describe (numbers):\n", df.describe())
print("C2 - describe (text columns):\n", df.describe(include="object"))

C2 - head:
    student_id             name          city           subject  marks  \
0        1014     Nisha Gurung       POKHARA         Databases   29.0   
1        1035  Ojaswi Adhikari      Birgunj         Statistics   52.0   
2        1036    Prakash Karki        DHARAN         Databases   71.0   
3        1019    Tara Shrestha    Kathmandu   Machine Learning   76.0   
4        1001   Aarav Shrestha     kathmandu         Databases   93.0   

   attendance_percent   exam_date  
0                81.4  2026-02-22  
1                80.4  2026-01-17  
2                89.8  2026-02-07  
3                81.4  2026-03-09  
4                92.2  2026-01-29  
C2 - tail(3):
      student_id           name           city           subject  marks  \
123        1016    Prakash Rai       LALITPUR        Statistics   90.0   
124        1037  Rita Shrestha      kathmandu  Machine Learning   79.0   
125        1015   Ojaswi Thapa    Biratnagar     Linear Algebra   70.0   

     attendance_perce

/tmp/ipykernel_46251/1076361903.py:8: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  print("C2 - describe (text columns):\n", df.describe(include="object"))


**C2 - three problems I can see:**
1. `marks` has a max of 150 in `describe()`, but marks should not go above 100 - some values are wrong.
2. `city` shows 24 unique values in `describe(include="object")`, even though there should only be a handful of real cities - the same city is spelled in different ways (capitals, lowercase, extra spaces).
3. `info()` shows `marks` and `attendance_percent` have fewer non-null rows than the total row count, so both columns have missing values.

## C3 - Selecting rows and columns

In [4]:
marks_as_series = df["marks"]
marks_as_dataframe = df[["marks"]]
print("C3 - type of df['marks']:", type(marks_as_series))
print("C3 - type of df[['marks']]:", type(marks_as_dataframe))
# df["marks"] with single brackets gives a Series (1 column of data).
# df[["marks"]] with double brackets gives a DataFrame (a table with 1 column).

loc_slice = df.loc[0:4, ["name", "subject", "marks"]]
iloc_slice = df.iloc[0:5, 1:4]
print("C3 - loc rows returned:", len(loc_slice))
print("C3 - iloc rows returned:", len(iloc_slice))
# loc uses LABELS and includes both ends, so 0:4 means rows 0,1,2,3,4 (5 rows).
# iloc uses POSITIONS like normal python slicing, so 0:5 stops before 5,
# giving rows 0,1,2,3,4 (also 5 rows here, but for a different reason).

C3 - type of df['marks']: <class 'pandas.Series'>
C3 - type of df[['marks']]: <class 'pandas.DataFrame'>
C3 - loc rows returned: 5
C3 - iloc rows returned: 5


## C4 - Boolean filtering

In [5]:
high_python_marks = df[(df["marks"] > 80) & (df["subject"] == "Python Programming")]
print("C4a - rows with marks > 80 in Python Programming:", len(high_python_marks))

kathmandu_or_pokhara = df[(df["city"] == "Kathmandu") | (df["city"] == "Pokhara")]
print("C4b - rows from Kathmandu or Pokhara (before cleaning city column):", len(kathmandu_or_pokhara))
# this number is too low right now because city has messy spelling -
# we will fix that in C5 and this count will go up.

high_python_marks_query = df.query("marks > 80 and subject == 'Python Programming'")
print("C4c - same result using query():", high_python_marks.equals(high_python_marks_query))
# Inside [] filters, and/or only work on single True/False values, not on a
# whole column of them, so we need & and | with brackets around each part.
# query() reads the condition as a text expression instead, so plain
# english-style and/or works there.

C4a - rows with marks > 80 in Python Programming: 3
C4b - rows from Kathmandu or Pokhara (before cleaning city column): 7
C4c - same result using query(): True


## C5 - Cleaning inconsistent text

In [6]:
print("C5 - unique city spellings before cleaning:", df["city"].nunique())
print("C5 - counts before cleaning:\n", df["city"].value_counts())

df["city"] = df["city"].str.strip().str.title()

print("C5 - unique city spellings after cleaning:", df["city"].nunique())
print("C5 - counts after cleaning:\n", df["city"].value_counts())

kathmandu_or_pokhara_clean = df[(df["city"] == "Kathmandu") | (df["city"] == "Pokhara")]
print("C5 - C4b rows now:", len(kathmandu_or_pokhara_clean), "compared to before:", len(kathmandu_or_pokhara))

C5 - unique city spellings before cleaning: 24
C5 - counts before cleaning:
 city
Biratnagar       10
  Pokhara         9
KATHMANDU         8
LALITPUR          8
kathmandu         7
  Kathmandu       6
Birgunj           6
BIRGUNJ           6
Dharan            6
  Dharan          6
pokhara           6
  Lalitpur        5
  Biratnagar      5
Lalitpur          5
Pokhara           5
dharan            4
biratnagar        4
birgunj           4
lalitpur          4
  Birgunj         3
BIRATNAGAR        3
POKHARA           2
DHARAN            2
Kathmandu         2
Name: count, dtype: int64
C5 - unique city spellings after cleaning: 6
C5 - counts after cleaning:
 city
Kathmandu     23
Pokhara       22
Biratnagar    22
Lalitpur      22
Birgunj       19
Dharan        18
Name: count, dtype: int64
C5 - C4b rows now: 45 compared to before: 7


## C6 - Missing values

In [7]:
print("C6 - missing values per column:\n", df.isna().sum())

mean_attendance = df["attendance_percent"].mean()
df["attendance_percent"] = df["attendance_percent"].fillna(mean_attendance)
print("C6 - row count after filling attendance (should be unchanged):", len(df))

print("C6 - row count before dropping missing marks:", len(df))

C6 - missing values per column:
 student_id             0
name                   0
city                   0
subject                0
marks                  4
attendance_percent    10
exam_date              0
dtype: int64
C6 - row count after filling attendance (should be unchanged): 126
C6 - row count before dropping missing marks: 126


**C6 - was mean-filling attendance a good idea?**
It is an okay quick fix because attendance is a percentage that stays in a fairly narrow range (52 to 99), so using the average does not create any crazy outlier. But it is not perfect - it hides the fact that we do not actually know those students' real attendance, and it can make later comparisons (like attendance vs marks) look weaker than they really are.

## C7 - Duplicates

In [8]:
duplicate_rows = df.duplicated()
print("C7 - number of fully duplicated rows:", duplicate_rows.sum())
print(df[duplicate_rows].head())

shape_before = df.shape
df = df.drop_duplicates(keep="first")
print("C7 - shape before:", shape_before, "shape after:", df.shape)

same_student_subject = df.duplicated(subset=["student_id", "subject"]).sum()
print("C7 - duplicates on [student_id, subject]:", same_student_subject)
# This subset matters more than exact duplicates because one student should
# only appear once per subject - if student_id + subject repeats, that means
# conflicting exam records for the same exam, which is a real data problem.

# now that duplicates are handled, drop the rows with missing marks
# (doing this after de-duplication so duplicate rows are checked against
# the full data, matching the 126 -> 120 checkpoint)
before_dropping_marks = len(df)
df = df.dropna(subset=["marks"])
print("C7 - rows before dropping missing marks:", before_dropping_marks, "after:", len(df))

C7 - number of fully duplicated rows: 6
     student_id             name        city             subject  marks  \
37         1035  Ojaswi Adhikari     Birgunj  Python Programming   80.0   
90         1031   Kiran Shrestha   Kathmandu           Databases   44.0   
108        1015     Ojaswi Thapa  Biratnagar  Python Programming   39.0   
109        1002     Bibek Gurung     Pokhara  Python Programming   48.0   
113        1025    Esha Shrestha   Kathmandu          Statistics   94.0   

     attendance_percent   exam_date  
37            73.300000  2026-02-21  
90            86.100000  2026-02-07  
108           82.100000  2026-02-06  
109           81.300000  2026-03-13  
113           74.413793  2026-03-17  
C7 - shape before: (126, 7) shape after: (120, 7)
C7 - duplicates on [student_id, subject]: 0
C7 - rows before dropping missing marks: 120 after: 116


## C8 - Dates

In [9]:
print("C8 - exam_date dtype before:", df["exam_date"].dtype)
df["exam_date"] = pd.to_datetime(df["exam_date"])
print("C8 - exam_date dtype after:", df["exam_date"].dtype)

df["exam_month"] = df["exam_date"].dt.month
df["exam_weekday"] = df["exam_date"].dt.day_name()

exams_after_cutoff = df[df["exam_date"] > "2026-02-15"]
print("C8 - exams held after 2026-02-15:", len(exams_after_cutoff))

C8 - exam_date dtype before: str
C8 - exam_date dtype after: datetime64[us]
C8 - exams held after 2026-02-15: 59


## C9 - Sorting and renaming

In [10]:
sorted_df = df.sort_values(by=["subject", "marks"], ascending=[True, False])
print("C9 - top rows after sorting:\n", sorted_df.head())

renamed_df = df.rename(columns={"attendance_percent": "attendance"})
print("C9 - original columns (unchanged):", df.columns.tolist())
print("C9 - renamed columns:", renamed_df.columns.tolist())

df = df.sort_index()
print("C9 - back to original row order, first few index values:", df.index[:5].tolist())

C9 - top rows after sorting:
      student_id            name        city    subject  marks  \
42         1027      Gita Thapa  Biratnagar  Databases   96.0   
48         1012     Laxmi Karki      Dharan  Databases   94.0   
4          1001  Aarav Shrestha   Kathmandu  Databases   93.0   
100        1022       Bibek Rai    Lalitpur  Databases   92.0   
21         1015    Ojaswi Thapa  Biratnagar  Databases   87.0   

     attendance_percent  exam_date  exam_month exam_weekday  
42                 73.6 2026-02-25           2    Wednesday  
48                 79.5 2026-01-22           1     Thursday  
4                  92.2 2026-01-29           1     Thursday  
100                89.4 2026-02-17           2      Tuesday  
21                 91.4 2026-01-15           1     Thursday  
C9 - original columns (unchanged): ['student_id', 'name', 'city', 'subject', 'marks', 'attendance_percent', 'exam_date', 'exam_month', 'exam_weekday']
C9 - renamed columns: ['student_id', 'name', 'city', 'su

## C10 - Derived columns

In [11]:
# marks of 150 are impossible, so I am removing those rows - keeping them
# would throw off the pass rate and grade counts with a fake value.
df = df[df["marks"] <= 100]

df["passed"] = df["marks"] >= 40
df["grade"] = np.where(df["marks"] >= 85, "A",
              np.where(df["marks"] >= 70, "B",
              np.where(df["marks"] >= 40, "C", "F")))

pass_rate = df["passed"].astype(int).mean() * 100
print(f"C10 - overall pass rate: {pass_rate:.1f}%")
print("C10 - grade counts:\n", df["grade"].value_counts())

C10 - overall pass rate: 78.9%
C10 - grade counts:
 grade
C    53
F    24
A    21
B    16
Name: count, dtype: int64


## C11 - Types and memory

In [12]:
memory_before = df[["subject", "city"]].memory_usage(deep=True)
print("C11 - memory before (bytes):\n", memory_before)

df["subject"] = df["subject"].astype("category")
df["city"] = df["city"].astype("category")

memory_after = df[["subject", "city"]].memory_usage(deep=True)
print("C11 - memory after (bytes):\n", memory_after)

percent_saved = (1 - memory_after.sum() / memory_before.sum()) * 100
print(f"C11 - percent saved: {percent_saved:.1f}%")
# category type stores each unique text value only once, and just uses a
# small number to point to it for every row, instead of repeating the full
# text every time. This saves a lot of memory when a column has few unique
# values repeated many times (like city or subject), but it would not help
# on a column where almost every value is different, like a name or an id.

C11 - memory before (bytes):
 Index       912
subject    7101
city       6487
dtype: int64
C11 - memory after (bytes):
 Index      912
subject    426
city       455
dtype: int64
C11 - percent saved: 87.6%


## C12 - Aggregation and a summary table

In [13]:
by_subject = df.groupby("subject")["marks"].agg(["mean", "max", "count"])
print("C12a - marks by subject:\n", by_subject)

by_city_subject = df.groupby(["city", "subject"])["marks"].mean()
print("C12b - mean marks by city and subject:\n", by_city_subject)

pivot = df.pivot_table(values="marks", index="city", columns="subject", aggfunc="mean")
print("C12c - pivot table:\n", pivot)

by_month = df.groupby("exam_month")["marks"].mean().sort_index()
print("C12d - mean marks per month:\n", by_month)

by_subject.to_csv("../outputs/summary.csv")
print("C12 - saved to outputs/summary.csv")

C12a - marks by subject:
                          mean   max  count
subject                                   
Databases           57.000000  96.0     23
Linear Algebra      51.166667  93.0     24
Machine Learning    59.809524  91.0     21
Python Programming  58.454545  98.0     22
Statistics          67.166667  95.0     24
C12b - mean marks by city and subject:
 city        subject           
Biratnagar  Databases             80.666667
            Linear Algebra        61.666667
            Machine Learning      65.600000
            Python Programming    52.166667
            Statistics            67.666667
Birgunj     Databases             53.000000
            Linear Algebra        59.800000
            Machine Learning      60.333333
            Python Programming    70.666667
            Statistics            60.000000
Dharan      Databases             63.200000
            Linear Algebra        47.666667
            Machine Learning      71.333333
            Python Programming

**C12 - what the month trend shows:** the mean marks across months 1, 2 and 3 stay close to each other with no clear upward or downward pattern, so there is no strong sign that students did better or worse as the term went on.

## Bonus +3 - Where my cleaning could mislead (150-250 words)

Two of my cleaning choices could change what the C12 summary seems to show.

First, filling missing `attendance_percent` values with the overall mean assumes those students had "normal" attendance. If the students who forgot to log attendance were actually the ones who missed more classes, I have quietly nudged their numbers up to look average. That would make any future comparison between attendance and marks look weaker than it really is, since the real low-attendance cases got hidden inside the average.

Second, dropping the two rows with an impossible mark of 150 instead of, say, capping them at 100 removes those students completely from every C12 table. If those two rows happened to belong to a subject with fewer rows overall (like Machine Learning), removing them shifts that subject's mean more than it would for a subject with lots of rows, simply because there is less data to smooth it out.

Both choices are reasonable and common ways to clean data, but neither one is completely neutral - they both quietly shape the numbers before they ever reach the groupby and pivot tables in C12, so it is worth remembering these were judgement calls, not exact facts about the students.